In [1]:
import os
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

load_dotenv("../.env")
GEMINI_API_KEY= os.getenv("GEMINI_API_KEY")

In [2]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex

llm= GoogleGenAI(model= "models/gemini-2.0-flash", api_key= GEMINI_API_KEY)
embed_model= GoogleGenAIEmbedding(model_name= "models/gemini-embedding-2-preview", api_key= GEMINI_API_KEY)

Settings.llm= llm
Settings.embed_model= embed_model
Settings.chunk_size= 512

In [3]:
documents= SimpleDirectoryReader("../data").load_data()
index= VectorStoreIndex.from_documents(documents)

print(f"Advanced Research Baseline Ready!")

2026-03-22 10:58:31,857 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 10:58:32,809 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"


Advanced Research Baseline Ready!


In [4]:
query= "What exactly is the DevGuardian MCP Server and what are its core features?"

baseline_engine= index.as_query_engine(streaming= True)
response= baseline_engine.query(query)

print("Baseline Response")
for token in response.response_gen:
    print(token, end="", flush= True)

2026-03-22 10:58:33,620 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 10:58:33,652 - INFO - AFC is enabled with max remote calls: 10.


Baseline Response


2026-03-22 10:58:34,491 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


The DevGuardian MCP Server is the core server that communicates with compatible clients, such as Antigravity and Claude Desktop. It exposes multiple tools that perform tasks such as debugging errors, generating code, reviewing pull requests, and scanning repositories for security vulnerabilities. It also integrates with version control systems such as Git. Its autonomous engineering capability is powered by LangGraph-based agents. It can automatically generate Dockerfiles and Docker Compose configurations and create GitHub Actions workflows for continuous integration and testing. It also includes a security scanning module that detects potential credential leaks.


In [5]:
from llama_index.core.postprocessor import LLMRerank
from llama_index.core import QueryBundle

reranker= LLMRerank(
    choice_batch_size= 5,
    top_n= 3,
    llm= Settings.llm
)

In [6]:
advanced_engine= index.as_query_engine(
    streaming= True,
    similarity_top_k= 10,
    node_postprocessors= [reranker]
)

In [7]:
print(f"RERANKED RESPONSE (Top-k: 10 -> Rerank: 3)")
response= advanced_engine.query(query)

for token in response.response_gen:
    print(token, end="", flush= True)

RERANKED RESPONSE (Top-k: 10 -> Rerank: 3)


2026-03-22 10:58:35,611 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 10:58:35,627 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:36,764 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 10:58:36,773 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:37,975 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 10:58:37,988 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:38,820 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


DevGuardian MCP Server is an AI-powered coding assistant designed to automate stages of the software development lifecycle. It integrates Gemini 2.0 Flash with a multi-agent architecture to perform debugging, code generation, testing, and security scanning. It analyzes project structures before generating code and includes a three-agent pipeline consisting of a coder, tester, and reviewer that collaboratively generate production-ready code. The system also integrates DevOps automation, GitHub pull request review, and security checks.


In [8]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever

nodes= list(index.docstore.docs.values())

bm25_retriever= BM25Retriever.from_defaults(
    nodes= nodes,
    similarity_top_k= 5
)

2026-03-22 10:58:39,364 - DEBUG - Building index from IDs objects


resource module not available on Windows


In [9]:
hybrid_retriever= QueryFusionRetriever(
    [
        index.as_retriever(similarity_top_k= 5),
        bm25_retriever
    ],
    num_queries= 1,
    use_async= True
)

In [10]:
from llama_index.core.query_engine import RetrieverQueryEngine

hybrid_query_engine = RetrieverQueryEngine.from_args(
    retriever=hybrid_retriever,
    node_postprocessors=[reranker],
    streaming=True
)

print("HYBRID SEARCH SYSTEM READY!")

HYBRID SEARCH SYSTEM READY!


In [11]:
response = hybrid_query_engine.query("What are the key takeaways from the DevGuardian docs?")
for token in response.response_gen:
    print(token, end="", flush=True)

2026-03-22 10:58:40,116 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:43,209 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 10:58:43,221 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:44,651 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


The DevGuardian MCP Server automates critical development tasks by combining multi-agent collaboration, automated testing, and security mechanisms. It reduces development time, improves code quality, and ensures software systems are reliable and secure. The system also includes DevOps automation features like generating Dockerfiles and GitHub Actions workflows, along with a security scanning module to prevent credential leaks. It can operate locally, leveraging cloud-based AI when needed.


In [12]:
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine

hyde= HyDEQueryTransform(include_original= True)

hyde_query_engine= TransformQueryEngine(hybrid_query_engine, hyde)

print("HYDE ADVANCED ENGINE READY")

HYDE ADVANCED ENGINE READY


In [13]:
test_query= "Tall me about the code generation part."
response= hyde_query_engine.query(test_query)

print(f"Original Query: {test_query}")
print("-" * 30)
for token in response.response_gen:
    print(token, end="", flush= True)

2026-03-22 10:58:45,062 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:48,278 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 10:58:50,961 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:54,967 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 10:58:54,978 - INFO - AFC is enabled with max remote calls: 10.


Original Query: Tall me about the code generation part.
------------------------------


2026-03-22 10:58:55,800 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


The system uses the Gemini 2.0 Flash model as its primary reasoning engine for understanding code and generating intelligent responses. The Coder Agent generates new code based on instructions, and the Tester Agent automatically generates unit tests to validate functionality. The Reviewer Agent then evaluates the generated code to ensure that it meets quality, security, and performance standards. The system can also analyze code differences and generate commit messages.


In [14]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine

individual_query_tool= QueryEngineTool(
    query_engine=hybrid_query_engine, 
    metadata=ToolMetadata(
        name="devguardian_docs",
        description="The technical documentation for DevGuardian MCP Server"
    )
)

In [15]:
from llama_index.core.question_gen import LLMQuestionGenerator
from llama_index.core.query_engine import SubQuestionQueryEngine

# 1. Force use of Gemini for sub-question generation
question_gen = LLMQuestionGenerator.from_defaults(llm=Settings.llm)

# 2. Build the agent with the explicit question generator
research_agent = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[individual_query_tool],
    question_gen=question_gen, # <--- This is the key fix!
    use_async=True
)

print("MULTI-STEP RESEARCH AGENT READY (Gemini Powered)")

MULTI-STEP RESEARCH AGENT READY (Gemini Powered)


In [16]:
complex_query = "Summarize the security features and then explain how they relate to the DevOps automation tools."
response = research_agent.query(complex_query)
print(f"Master Query: {complex_query}")
print("-" * 30)
print(response)

2026-03-22 10:58:56,165 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:58,200 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


Generated 3 sub questions.
[devguardian_docs] Q: What are the security features of DevGuardian MCP Server?
[devguardian_docs] Q: What DevOps automation tools are compatible with DevGuardian MCP Server?
[devguardian_docs] Q: How do the security features of DevGuardian MCP Server relate to the compatible DevOps automation tools?


2026-03-22 10:58:58,883 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:59,286 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:58:59,564 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:59:00,496 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 10:59:00,511 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 10:59:00,992 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 10:59:01,229 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 10:59:01,772 - INFO - AFC is enabled with max remote calls: 10.


[devguardian_docs] A: DevGuardian has a security scanning module that detects potential credential leaks such as API keys, tokens, and passwords, preventing accidental exposure of sensitive information when developers push code to remote repositories. The system can also operate locally on a developer's machine, ensuring that the source code remains secure while leveraging cloud-based AI capabilities only when necessary.



2026-03-22 10:59:02,748 - INFO - AFC is enabled with max remote calls: 10.


[devguardian_docs] A: DevGuardian MCP Server includes DevOps automation features such as the automatic generation of Dockerfiles and Docker Compose configurations based on project dependencies, and the creation of GitHub Actions workflows for continuous integration and testing.



2026-03-22 10:59:03,774 - INFO - AFC is enabled with max remote calls: 10.


[devguardian_docs] A: DevGuardian includes a security scanning module that detects potential credential leaks, preventing accidental exposure of sensitive information when developers push code to remote repositories. Additionally, the system can automatically generate Docker configuration files and continuous integration pipelines, simplifying project deployment and infrastructure management.



2026-03-22 10:59:05,803 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


Master Query: Summarize the security features and then explain how they relate to the DevOps automation tools.
------------------------------
The security features include a scanning module that detects potential credential leaks, which prevents the exposure of sensitive information when code is pushed to remote repositories. These features relate to the DevOps automation tools because the system can automatically generate Docker configuration files and continuous integration pipelines, which simplifies project deployment and infrastructure management.



In [17]:
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.retrievers import RecursiveRetriever
from llama_index.core import StorageContext

# 1. Create a "Hierarchical" Node Parser
# It will make Parent chunks (1024) -> Child chunks (256) -> Leaf chunks (128)
node_parser= HierarchicalNodeParser.from_defaults(
    chunk_sizes= [1024, 256, 128]
)

In [18]:
# 2. Re-parse your documents
nodes = node_parser.get_nodes_from_documents(documents)
leaf_nodes = get_leaf_nodes(nodes) # These are the tiny "Sniper" nodes we will search

In [19]:
# 3. Create a special index with BOTH parents and leaves
# We store ALL nodes but we only vector-index the leaves to keep it sharp
storage_context = StorageContext.from_defaults()
storage_context.docstore.add_documents(nodes)
recursive_index = VectorStoreIndex(
    leaf_nodes, # Indexing the smallest nodes
    storage_context=storage_context
)

2026-03-22 10:59:25,187 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 10:59:26,195 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 10:59:27,258 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 10:59:28,251 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 10:59:29,251 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 10:59:30,219 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batch

In [20]:
# 4. Build the Recursive Retriever
# This will find a "Leaf" but it will "Recurse" up to the "Parent"
recursive_retriever= RecursiveRetriever(
    root_id= "vector",
    retriever_dict= {"vector": recursive_index.as_retriever(similarity_top_k= 5)},
    node_dict= {node.node_id: node for node in nodes},
    verbose= True
)

In [21]:
# 5. Build the Final Engine
recursive_query_engine= RetrieverQueryEngine.from_args(
    retriever= recursive_retriever,
    node_postprocessors= [reranker],
    streaming= True
)

print("RECURSIVE (PARENT-CHILD) READY")

RECURSIVE (PARENT-CHILD) READY


In [22]:
# 6. Test with a technical question
response= recursive_query_engine.query("Explain the security scanner module in depth.")
for token in response.response_gen:
    print(token, end="", flush= True)

Retrieving with query id None: Explain the security scanner module in depth.


2026-03-22 11:12:55,128 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-03-22 11:12:55,156 - INFO - AFC is enabled with max remote calls: 10.


Retrieving text node: Security  is  another  critical  aspect  of  the  implementation.  DevGuardian  includes  a  security  scanning  module  that  detects  potential  credential  leaks  such  as  API  keys,  tokens,  and  passwords.
Retrieving text node: Additional  modules  are  implemented  to  scan  repositories  for  potential  security  vulnerabilities  and  accidental  exposure  of  API  keys  or  credentials.    RESULTS  AND  DISCUSSION:  The  DevGuardian  MCP  Server  successfully  demonstrates  the  concept  of  an  autonomous  AI-powered  engineering  assistant.
Retrieving text node: This  module  prevents  accidental  exposure  of  sensitive  information  when  developers  attempt  to  push  code  to  remote  repositories.
Retrieving text node: While  this  helps  prevent  accidental  leaks,  deeper  vulnerability  scanning  for  issues  such  as  injection  attacks  or  insecure  dependencies  could  be  improved  in  future  versions.
Retrieving text node: Another  impor

2026-03-22 11:12:56,248 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2026-03-22 11:12:56,261 - INFO - AFC is enabled with max remote calls: 10.
2026-03-22 11:12:57,088 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:streamGenerateContent?alt=sse "HTTP/1.1 200 OK"


The security scanner module detects potential credential leaks, such as API keys, tokens, and passwords, and prevents accidental exposure of sensitive information when developers attempt to push code to remote repositories. Additional modules scan repositories for potential security vulnerabilities and accidental exposure of API keys or credentials.
